In [1]:
import os
os.environ["QT_MULTIMEDIA_PREFERRED_PLUGINS"] = "windowsmediafoundation"
import subprocess
from pathlib import Path
import imageio_ffmpeg
from PyQt5.QtCore import QUrl, Qt, QSize
from PyQt5.QtWidgets import QApplication
from qfluentwidgets.multimedia import VideoWidget
from qfluentwidgets import ClickableSlider, setFont, themeColor
FILE = Path(r"F:\摄像头记录\20260904_132624_tp00006.mp4")


📢 Tips: QFluentWidgets Pro is now released. Click https://qfluentwidgets.com/pages/pro to learn more about it.



In [2]:
video = FILE.with_suffix(".mkv")
if not video.exists():
    temporary = video.with_suffix(".part.mkv")
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(), "-hide_banner", "-loglevel", "error", "-nostdin", "-y",
                    "-copyts", "-start_at_zero", "-i", str(FILE), "-map", "0:v:0", "-map", "0:a:0?",
                    "-c", "copy", str(temporary)],
                   check=True, creationflags=subprocess.CREATE_NO_WINDOW)
    temporary.replace(video)

In [ ]:
get_ipython().run_line_magic("gui", "qt5")
app = QApplication.instance() or QApplication([])
if "window" in globals():
    window.close()
    window.deleteLater()
window = VideoWidget()
window.setWindowTitle(FILE.name)
window.videoItem.nativeSizeChanged.connect(
    lambda size: window.resize(size.scaled(960, app.primaryScreen().availableGeometry().height() * .8, Qt.KeepAspectRatio).toSize()))
bar = window.playBar
bar.setFixedHeight(132)
for button in (bar.playButton, bar.skipBackButton, bar.skipForwardButton, bar.volumeButton):
    button.setFixedSize(44, 44)
    button.setIconSize(QSize(26, 26))
for label in (bar.currentTimeLabel, bar.remainTimeLabel):
    setFont(label, 20)
old_slider = bar.progressSlider
bar.progressSlider = ClickableSlider(Qt.Horizontal, bar)
bar.progressSlider.setFixedHeight(36)
accent = themeColor().name()
bar.progressSlider.setStyleSheet(f"QSlider::groove:horizontal {{height: 8px; background: lightgray; border-radius: 4px;}} QSlider::sub-page:horizontal {{background: {accent}; border-radius: 4px;}} QSlider::handle:horizontal {{width: 26px; margin: -12px 0; background: white; border: 3px solid {accent}; border-radius: 16px;}}")
bar.vBoxLayout.replaceWidget(old_slider, bar.progressSlider)
window.player.durationChanged.disconnect(old_slider.setMaximum)
window.player.durationChanged.connect(bar.progressSlider.setMaximum)
bar.progressSlider.sliderMoved.connect(window.player.setPosition)
bar.progressSlider.clicked.connect(window.player.setPosition)
old_slider.hide()
old_slider.deleteLater()
window.setVideo(QUrl.fromLocalFile(str(video.resolve())))
window.show()
window.play()